In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm import tqdm
tqdm.pandas()

from sklearn.metrics import classification_report, confusion_matrix

import sys
sys.path.append('../code/poseEvaluation/')

from rag import search

from typing import List, Annotated, Literal, Any
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from enum import Enum

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import AnyMessage, AIMessage, ToolMessage, HumanMessage
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnableLambda, RunnableWithFallbacks

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

from dotenv import load_dotenv
load_dotenv()

# 데이터베이스 설정 및 도구 정의를 위한 추가 임포트
from langchain_community.utilities import SQLDatabase
from langchain_community.tools.sql_database.tool import (
    QuerySQLDataBaseTool,
    InfoSQLDatabaseTool,
    ListSQLDatabaseTool,
)
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_tool_calling_agent
from langgraph.prebuilt import create_agent_executor

# 데이터베이스 다운로드 (WikiDocs의 내용 기반)
import requests
if not os.path.exists("Chinook.db"):
    url = "https://storage.googleapis.com/benchmarks-artifacts/chinook/Chinook.db"
    response = requests.get(url)
    if response.status_code == 200:
        with open("Chinook.db", "wb") as file:
            file.write(response.content)
        print("File downloaded and saved as Chinook.db")
    else:
        print(f"Failed to download the file. Status code: {response.status_code}")
else:
    print("Chinook.db already exists.")

# LLM 설정
llm = ChatOpenAI(
    temperature=0.2,
    model="gpt-4o-mini",
)

# 데이터베이스 인스턴스 생성
db = SQLDatabase.from_uri("sqlite:///Chinook.db")

# SQL 도구 정의
list_tables_tool = ListSQLDatabaseTool(db=db)
get_schema_tool = InfoSQLDataBaseTool(db=db)
db_query_tool = QuerySQLDataBaseTool(db=db)
sql_tools = [list_tables_tool, get_schema_tool, db_query_tool]

# 오류 처리 함수
def handle_tool_error(state) -> dict:
    error = state.get("error")
    tool_calls = state["messages"][-1].tool_calls
    return {
        "messages": [
            ToolMessage(
                content=f"Here is the error: {repr(error)}\n\nPlease fix your mistakes.",
                tool_call_id=tc["id"],
            )
            for tc in tool_calls
        ]
    }

# 오류를 처리하고 에이전트에 오류를 전달하기 위한 ToolNode 생성
def create_tool_node_with_fallback(tools: list) -> RunnableWithFallbacks[Any, dict]:
    """
    Create a ToolNode with a fallback to handle errors and surface them to the agent.
    """
    return ToolNode(tools).with_fallbacks(
        [RunnableLambda(handle_tool_error)], exception_key="error"
    )

# SQL ToolNode 생성
sql_tool_node = create_tool_node_with_fallback(sql_tools)

print("Setup complete.")

### 1. 예상 질의
* 예상 질의는 90개로 구성
* 3개의 intent로 구분
    1. 복합 질의
        * 다단계의 논리적 처리 또는 다중 도구 호출이 필요한 질의(40개)
            * RAG + Text to SQL
            * 복잡한 Text to SQL
        * 하나의 도구 호출로 완료될 수 없으며, 중간 결과의 통합, 비교, 추론을 위해 plan and excute 구조 기반 답변
        * 예시
            * 나의 용상 동작과 인상 동작에서 공통적으로 발생하는 고질적인 문제점이 무엇이야?
            * 내가 올린 영상 중 평균 점수 80점 미만인 영상들의 가장 흔한 오류 부위는 무엇이야?
            * 내가 올린 영상 중 평균 점수 95점 이상인 영상들의 공통적인 특징을 분석하고, 기술 구조를 참조하여 요약해줘.
    2. 단순 질의
        * 하나의 도구 호출이 필요한 질의(30개)
            * 단순 RAG
            * 단순 Text to SQL
        * 예시
            * 지면 반발력(Ground Reaction Force)이 역도 동작에 어떻게 활용되나요?
            * 스내치(Snatch) 동작에서 바벨을 받는 시점에 대한 기술적 조언을 해주세요.
            * 가장 최근에 올린 영상의 전체 점수는 몇점이야?
    3. 예외 질의
        * 서비스의 범위를 벗어나거나(일반 대화, 날씨 등), 현재 지원되지 않는 기능(영상 편집, 계정 관리 등)에 대한 질의(20개)
        * 기존 저장된 메시지로 답변 출력력
        * 예시
            * 파이썬으로 DTW 알고리즘을 구현하는 코드를 알려주세요.
            * 가장 가까운 역도 체육관 위치를 찾아주세요.
            * 영상 ID 500을 삭제해 주세요.

In [11]:
query_path = '..\\data\\rag\\intent_queries_90.csv'

# query_df = pd.read_csv(query_path) # 주석 처리: 데이터 파일이 없으므로 실행 불가

In [5]:
# test_df = query_df.groupby('category').sample(frac=0.2)
# query_df['type'] = np.nan
# query_df.loc[~query_df.index.isin(test_df.index), 'type'] = 'validation'
# query_df.loc[query_df.index.isin(test_df.index), 'type'] = 'test'

# query_df.to_csv(query_path, encoding='utf-8-sig', index=False) # 주석 처리: 데이터 파일이 없으므로 실행 불가

### 2. State

In [ ]:
class State(TypedDict):
    intent: Annotated[Literal["COMPLEX", "SIMPLE", "INAPPROPRIATE"], "Intent Category"]
    messages: Annotated[list, add_messages]
    # SQL 에이전트의 중간 상태를 저장하기 위한 필드 추가
    sql_query: str
    sql_result: str

In [8]:
graph_builder = StateGraph(State)



### 3. Intent Analyze
#### 개요
* 사용자 쿼리의 intent(의도) 분석 및 query rewrite 작업
* intent 종류:
    1. COMPLEX: 다단계 처리 또는 멀티툴 사용 요구 질의 (plan & execute sub-graph 처리)
    2. SIMPLE: 단일 응답으로 해결 가능한 간단 질의
    3. INAPPROPRIATE: 서비스 범위 외 또는 부적절 질의

#### 실험 결과
* intent 분석용 프롬프트 기법 비교(COT, COD)

* COT 기법 성능
    * 전체 정확도: 80.0%
    * 전체 Precision: 83.53%, Recall: 80.00%, F1-Score: 76.22%
    * COMPLEX: Precision 81.63%, Recall 100.00%, F1-Score 89.89%
    * SIMPLE: Precision 100.00%, Recall 40.00%, F1-Score 57.14%
    * INAPPROPRIATE: Precision 68.97%, Recall 100.00%, F1-Score 81.63%
    * 특징: COMPLEX와 INAPPROPRIATE는 Recall 100%로 완벽히 포착하나, Precision이 낮아 false positive 발생
    * SIMPLE의 Recall 40%로 SIMPLE 쿼리의 대부분을 COMPLEX로 오분류하는 경향

* COD 기법 성능
    * 전체 정확도: 80.0%
    * 전체 Precision: 86.80%, Recall: 73.06%, F1-Score: 72.45%
    * COMPLEX: Precision 97.37%, Recall 92.50%, F1-Score 94.87%
    * SIMPLE: Precision 63.04%, Recall 96.67%, F1-Score 76.32%
    * INAPPROPRIATE: Precision 100.00%, Recall 30.00%, F1-Score 46.15%
    * 특징: COMPLEX와 SIMPLE에서 균형잡힌 성능, INAPPROPRIATE의 Recall 30%로 오분류 위험

* 분석 및 결론
    * 두 기법 모두 전체 정확도 80.0%로 동일
    * COT: INAPPROPRIATE Recall 100%로 부적절 쿼리를 확실히 차단하지만, SIMPLE 쿼리의 대부분을 COMPLEX로 오분류
    * COD: COMPLEX와 SIMPLE에서 더 균형잡힌 성능을 보이지만, INAPPROPRIATE Recall 30%로 부적절 쿼리를 놓칠 위험 높음
    * 서비스 관점: INAPPROPRIATE 분류 실패는 사용자 경험에 직접적 영향 → COT 기법 선택 권장
    * COT의 SIMPLE→COMPLEX 오분류는 서비스 관점에서 큰 문제 아님 (더 많은 처리 리소스 사용하지만 결과는 제공)

In [4]:
llm = ChatOpenAI(
    temperature=0.2,
    model="gpt-4o-mini",
)

In [ ]:
class Category(str, Enum):
    """Defines the query processing categories."""
    COMPLEX = "COMPLEX"
    SIMPLE = "SIMPLE"
    INAPPROPRIATE = "INAPPROPRIATE"

class Intent(BaseModel):
    """Schema containing the user query’s intent, processing category, and rewritten query."""
    category: Category = Field(
        description="Query processing category. Must be one of 'COMPLEX', 'SIMPLE', or 'INAPPROPRIATE'."
    )
    query_rewrite: str = Field(
        description="A rewritten version of the original query to make it easier for downstream Agents to process. If the category is INAPPROPRIATE, contains a rejection message."
    )

parser = PydanticOutputParser(pydantic_object=Intent)

#### 3-01. COT

In [ ]:
intent_template = """
[SYSTEM ROLE]
You are the Intent Router for a weightlifting analysis service. Your task is to analyze the user's query and determine the appropriate processing Category and a Rewritten Query for the next processing step.

[CATEGORY DEFINITIONS]
1. COMPLEX: Requires multi-step processing, multi-tool usage (SQL + RAG/LLM Inference), or logical comparison/analysis (Plan-and-Execute).
2. SIMPLE: Requires a single tool call (RAG or Single SQL) (Execute Only).
3. INAPPROPRIATE: Outside the service scope (OUT_OF_SCOPE) or unsupported feature (UNHANDLED_TOOL).

[OUTPUT FORMAT]
{format}

[FEW-SHOT EXAMPLE]

User Query: "What are the common chronic faults in my Snatch and Clean & Jerk movements?"

Thought:
1. Analysis: The user is asking for a comparison and synthesis of data from two different lift types (Snatch and C&J).
2. Tooling: This requires multiple SQL queries (Snatch data, C&J data), followed by LLM inference to find commonalities, and finally RAG for technical explanation.
3. Conclusion: This is a multi-step process requiring planning. The Category is COMPLEX.

Output:
{{
  "category": "COMPLEX",
  "query_rewrite": "Calculate the overall average DTW score of all user videos and compare it with the DTW score of the most recently uploaded video."
}}

[USER QUERY]
{user_query}
"""

In [ ]:
prompt = PromptTemplate.from_template(template=intent_template)
prompt = prompt.partial(format=parser.get_format_instructions())
cot_chain = prompt | llm

In [ ]:
def route_intent(state: State) -> str:
    """Intent 분석 결과를 바탕으로 다음 노드를 결정합니다."""
    intent = state["intent"]
    if intent == "COMPLEX":
        return "complex_flow"
    elif intent == "SIMPLE":
        # 단순 질의는 SQL 에이전트 또는 RAG 에이전트로 라우팅될 수 있습니다.
        # 여기서는 Text-to-SQL 에이전트 구현에 초점을 맞춥니다.
        return "sql_agent_simple"
    else: # INAPPROPRIATE
        return "end_with_rejection"

### 4. SQL Agent (Simple Flow)
단순 질의 중 SQL 관련 질의를 처리하는 에이전트입니다. 이 에이전트는 SQL 쿼리 생성, 실행, 결과 요약의 단계를 거칩니다.

In [ ]:
system_prompt = (
    "You are an expert SQL agent. Your goal is to answer user questions by generating and executing SQL queries against the database."
    "The database contains tables: {table_names}."
    "You must first use the `list_tables_sql_database` tool to see all available tables."
    "Then, use the `schema_sql_database` tool to get the DDL for the relevant tables."
    "Finally, use the `query_sql_database` tool to execute the generated SQL query."
    "If the query fails, analyze the error and try to fix the query."
    "After successfully executing the query, summarize the result to answer the user's question."
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("placeholder", "{messages}"),
    ]
).partial(table_names=db.get_usable_table_names())

# SQL 에이전트 생성
sql_agent_runnable = create_tool_calling_agent(llm, sql_tools, prompt)

def run_sql_agent(state: State):
    """SQL 에이전트를 실행하고 결과를 상태에 저장합니다."""
    # LangGraph의 AgentExecutor를 사용하여 에이전트 실행
    # 여기서는 간단히 Runnable을 실행하는 것으로 대체합니다.
    # 실제 LangGraph에서는 create_agent_executor를 사용하여 서브 그래프를 구성합니다.
    
    # 에이전트 실행 (LangGraph 서브 그래프 없이 간단한 Runnable로 시뮬레이션)
    # messages는 HumanMessage와 AIMessage의 리스트입니다.
    result = sql_agent_runnable.invoke({"messages": state["messages"]})
    
    # 결과 메시지를 상태에 추가
    return {"messages": [result]}

### 5. Graph 구성
Intent 분석을 시작으로 단순 SQL 질의 처리 흐름을 구성합니다.

In [ ]:
# 1. 노드 추가
def analyze_intent(state: State):
    """사용자 쿼리의 의도를 분석합니다."""
    # 마지막 HumanMessage를 추출
    user_query = state["messages"][-1].content
    # Intent 분석 체인 실행
    intent_result = cot_chain.invoke({"user_query": user_query})
    
    # 결과 파싱
    try:
        parsed_intent = parser.parse(intent_result.content)
    except Exception:
        # 파싱 실패 시 INAPPROPRIATE로 처리
        parsed_intent = Intent(category=Category.INAPPROPRIATE, query_rewrite="Intent analysis failed.")
        
    # 상태 업데이트
    return {
        "intent": parsed_intent.category.value,
        "messages": [AIMessage(content=parsed_intent.query_rewrite)] # Rewritten Query를 AIMessage로 추가
    }

def end_with_rejection(state: State):
    """부적절 질의에 대한 답변을 생성합니다."""
    # Intent 분석 단계에서 생성된 거절 메시지를 최종 답변으로 사용
    final_message = state["messages"][-1].content
    return {"messages": [AIMessage(content=final_message)]}

# 노드 추가
graph_builder.add_node("analyze_intent", analyze_intent)
graph_builder.add_node("sql_agent_simple", run_sql_agent)
graph_builder.add_node("end_with_rejection", end_with_rejection)

# 2. 시작점 설정
graph_builder.set_entry_point("analyze_intent")

# 3. 조건부 엣지 추가
graph_builder.add_conditional_edges(
    "analyze_intent",
    route_intent,
    {
        "complex_flow": END, # 복합 질의는 현재 구현 범위 밖이므로 END로 처리
        "sql_agent_simple": "sql_agent_simple",
        "end_with_rejection": "end_with_rejection",
    },
)

# 4. 일반 엣지 추가
graph_builder.add_edge("sql_agent_simple", END)
graph_builder.add_edge("end_with_rejection", END)

# 5. 그래프 컴파일
app = graph_builder.compile()

print("LangGraph app compiled.")

### 6. 에이전트 실행 및 테스트

In [ ]:
def run_agent(query: str):
    """에이전트를 실행하고 최종 결과를 출력합니다."""
    # 초기 상태
    initial_state = {"messages": [HumanMessage(content=query)]}
    
    # 그래프 실행
    final_state = app.invoke(initial_state)
    
    # 최종 답변 추출
    final_answer = final_state["messages"][-1].content
    
    print(f"--- User Query ---\n{query}\n")
    print(f"--- Final Answer ---\n{final_answer}\n")
    
    return final_answer

# 테스트 쿼리 1: 단순 SQL 질의 (SIMPLE)
run_agent("앨범 수가 가장 많은 아티스트 5명을 알려줘.")

# 테스트 쿼리 2: 부적절 질의 (INAPPROPRIATE)
run_agent("오늘 날씨는 어때?")

# 테스트 쿼리 3: 복합 질의 (COMPLEX) - 현재 END로 라우팅됨
run_agent("가장 최근에 올린 영상의 전체 점수는 몇점이야? 그리고 그 점수가 내 평균 점수보다 높은지 알려줘.")